In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pyspark import SparkConf
from pyspark.sql import SparkSession
import pyspark.pandas as ps
import pyspark.sql.functions as F
import os

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [3]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import MahalanobisDistance
from hypex.encoders.encoders import DummyEncoder
from hypex.comparators import TTest, Chi2Test
from hypex.comparators.distances import MahalanobisDistance
from hypex.operators import Bias, MatchingMetrics
from hypex.analyzers import MatchingAnalyzer
from hypex.utils import SparkSessionCalculator
from hypex.config import DatasetConfig

In [4]:
import os
import time
from pyspark.sql import SparkSession
from hypex.utils.spark_config import SparkSessionCalculator

# --- 1. Настройки окружения для macOS (Важно!) ---
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация для калькулятора ---
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 4
MEMORY_PER_EXECUTOR_MB = 2048

# Параметры для калькулятора (на основе вашего датасета)
n_rows = 10_000
n_columns = 7
n_categorical = 1

# --- 3. Создание и использование калькулятора ---
calculator = SparkSessionCalculator(
    data_size_bytes=n_rows * n_columns * 8,  # Примерная оценка
    num_columns=n_columns,
    num_categorical_columns=n_categorical,
    target_executor_cores=CORES_PER_EXECUTOR,
    target_executor_memory_gb=MEMORY_PER_EXECUTOR_MB / 1024
)

# Получаем оптимальные настройки
optimal_settings = calculator.calculate_optimal_settings()

print("\n📊 Оптимальные настройки от калькулятора:")
print(f"  Executor instances: {optimal_settings.executor_instances}")
print(f"  Executor cores: {optimal_settings.executor_cores}")
print(f"  Executor memory: {optimal_settings.executor_memory}")
print(f"  Shuffle partitions: {optimal_settings.shuffle_partitions}")

# --- 4. Создание Spark-сессии с настройками калькулятора ---
MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"
print(f"\n🚀 Запуск в режиме: {MASTER_URL}")

builder = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Настройки драйвера
    .config("spark.driver.memory", "2g")
    # Настройки executor'ов из калькулятора
    .config("spark.executor.memory", f"{MEMORY_PER_EXECUTOR_MB}m")
    .config("spark.executor.cores", str(CORES_PER_EXECUTOR))
    .config("spark.executor.instances", str(NUM_EXECUTORS))
    .config("spark.executor.memoryOverhead", "384m")
    # Дополнительные настройки из калькулятора
    .config("spark.memory.fraction", str(optimal_settings.memory_fraction))
    .config("spark.sql.shuffle.partitions", str(optimal_settings.shuffle_partitions))
    .config("spark.serializer", optimal_settings.serializer)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true")
    .config("spark.python.worker.reuse", "false")
)

sp_s = builder.getOrCreate()
sp_s.sparkContext.setLogLevel("WARN")

# --- 5. Проверка конфигурации ---
print(f"\n✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")
print(f"Shuffle Partitions: {sp_s.conf.get('spark.sql.shuffle.partitions')}")

# Проверка количества экзекуторов
time.sleep(3)
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 6. Тест на распределение ---
def print_executor_info(iterator):
    import os
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

df = sp_s.range(0, 10, 1, 4)
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# sp_s.stop()  # Раскомментируйте, если нужно остановить сессию


📊 Оптимальные настройки от калькулятора:
  Executor instances: 4
  Executor cores: 4
  Executor memory: 4g
  Shuffle partitions: 10

🚀 Запуск в режиме: local-cluster[2, 4, 2048]


26/09/23 20:03:49 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.72.75 instead (on interface wlo1)
26/09/23 20:03:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 20:03:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



✅ Сессия создана.
Driver Memory Config: 2g
Executor Memory Config: 2048m
Shuffle Partitions: 10


📊 Активных экзекуторов (проверка через RDD): 2

🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 27288
Executor ID: Driver/Local, PID: 27296
Executor ID: Driver/Local, PID: 27292
Executor ID: Driver/Local, PID: 27300


In [5]:
n_rows = 10000
n = n_rows
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "feat_cat": np.random.choice(["A", "B", "C"], size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

In [6]:
"""
PYSPARK case
"""
# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
)

In [7]:
from hypex.config import MatchingConfig

os.environ["HYPEX_LOG_LEVEL"] = "DEBUG"
os.environ["HYPEX_LOG_FILE"] = "experiment.log"

MatchingConfig.FAISS_FIT_MODE = "sample"
# MatchingConfig.FAISS_CHUNK_SIZE = 512

In [8]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=True,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=10,
    encode_categories=True      # DummyEncoder включится автоматически
)
matching.experiment

2026-09-23 20:03:59 | DEBUG    | hypex.experiment | ▶ __init__
2026-09-23 20:03:59 | DEBUG    | hypex.experiment | ▶ __init__
2026-09-23 20:03:59 | DEBUG    | hypex.experiment | ▶ __init__
2026-09-23 20:03:59 | DEBUG    | hypex.experiment | ✓ __init__ completed in 0.000s
2026-09-23 20:03:59 | DEBUG    | hypex.experiment | ✓ __init__ completed in 0.001s
2026-09-23 20:03:59 | DEBUG    | hypex.experiment | ▶ __init__
2026-09-23 20:03:59 | DEBUG    | hypex.experiment | ✓ __init__ completed in 0.000s
2026-09-23 20:03:59 | DEBUG    | hypex.experiment | ✓ __init__ completed in 0.002s


Experiment(executors=[DummyEncoder(target_roles=Feature(), key=''), TypeCaster(dtype={<class 'int'>: <class 'float'>}, roles=[Feature(), Target(None)], downcasting=True, key=''), MahalanobisDistance(grouping_role=Treatment(None), key='', weights=None), FaissNearestNeighbors(n_neighbors=10, two_sides=True, test_pairs=False, grouping_role=Treatment(None), key='', faiss_mode='base'), Bias(grouping_role=Treatment(None), target_roles=[Target(None)], key=''), MatchingMetrics(grouping_role=Treatment(None), target_roles=[Target(None)], metric='ate', n_neighbors=10, key=''), MatchingAnalyzer(key=''), OnRoleExperiment(executors=[TTest(grouping_role=Treatment(None), target_roles=Target(None), baseline_role=AdditionalMatching(None), reliability=0.05, compare_by='matched_pairs', key='')], role=[Feature()], transformer=False, key='')], transformer=True, key='')

In [9]:
# for name, val in sp_s.sparkContext.getConf().getAll():
#     print(f"{name}: {val}")

In [10]:
from hypex.utils import SparkMemoryMonitor

monitor = SparkMemoryMonitor(
    spark=sp_s,
    heap_threshold_pct=80.0,
    offheap_threshold_mb=384,
    total_memory_limit_mb=384 + 2048,  # 8g heap + 18g overhead
    log_file="matching_memory_log.csv",
    alert_log_file="matching_alerts.csv",
)

In [11]:
# monitor.start(interval=5)
# try:
#     # 6. Запуск
#     print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

#     print("✅ Выполнено успешно!")
# except Exception as e:
#     # logger.error(f"Pipeline failed: {e}")
#     pass
# finally:
#     # Остановка и отчёт
#     monitor.stop()
#     monitor.print_report()
#     monitor.plot_memory("matching_memory.png")
# # print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
# # print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

2026-09-23 20:04:07 | DEBUG    | hypex.experiment | ▶ __init__
2026-09-23 20:04:07 | DEBUG    | hypex.experiment | ✓ __init__ completed in 0.000s


2026-09-23 20:04:07 | DEBUG    | hypex.experiment | ============================================================
2026-09-23 20:04:07 | DEBUG    | hypex.experiment | Spark Session Info:
2026-09-23 20:04:07 | DEBUG    | hypex.experiment |   Master: local-cluster[2, 4, 2048]
2026-09-23 20:04:07 | DEBUG    | hypex.experiment |   App name: LocalClusterTest
2026-09-23 20:04:07 | DEBUG    | hypex.experiment |   Driver memory: 2g
2026-09-23 20:04:07 | DEBUG    | hypex.experiment |   Executor memory: 2048m
2026-09-23 20:04:07 | DEBUG    | hypex.experiment |   Executor cores: 4
2026-09-23 20:04:07 | DEBUG    | hypex.experiment |   Executor instances: 8
2026-09-23 20:04:07 | DEBUG    | hypex.experiment |   Spark version: 3.5.1
2026-09-23 20:04:07 | DEBUG    | hypex.experiment | ============================================================
2026-09-23 20:04:07 | INFO     | hypex.experiment | ▶ Process started: DummyEncoder [spark]
2026-09-23 20:04:08 | DEBUG    | hypex.experiment | [DummyEncoder|spa

In [12]:
result_data.resume

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,0.09,0.22,0.68,-0.35,0.53
ATC,0.06,0.22,0.80,-0.38,0.49
ATE,0.07,0.22,0.75,-0.36,0.50


In [13]:
from hypex.utils import FaissIndexStorage

FaissIndexStorage.cleanup()

In [14]:
# 1. Clear DataFrame and SQL Catalog level caches
sp_s.catalog.clearCache()

# 2. Extract the internal Java Spark Context map of persistent RDDs
persistent_rdds = sp_s.sparkContext._jsc.getPersistentRDDs()

# 3. Forcefully unpersist every remaining RDD block sequentially
for rdd_id in list(persistent_rdds.keys()):
    persistent_rdds.get(rdd_id).unpersist(True)  # True forces a synchronous blocking wipe

print("All DataFrame and RDD caches have been forced out of storage.")


All DataFrame and RDD caches have been forced out of storage.


In [15]:
import shutil
import os
from pathlib import Path
import gc
import builtins

def cleanup_spark_local_dirs(sc):
    """Удаляет все временные файлы из локальных каталогов экзекьюторов."""
    local_dirs = sc.getConf().get("spark.local.dir", "").split(",")
    if not local_dirs[0]:
        # Дефолтный путь
        local_dirs = [os.path.join("/tmp", f"spark-{os.getlogin()}")]
    for d in local_dirs:
        d = Path(d)
        if d.exists():
            try:
                shutil.rmtree(d)
            except Exception as e:
                print(f"Не удалось удалить {d}: {e}")

def force_jvm_gc(sc, verbose=True):
    """Принудительный GC в JVM драйвера через Py4J."""
    if verbose:
        print("🧹 Запуск принудительного GC в JVM...")
    
    # 1. Принудительный вызов System.gc() в JVM драйвера
    sc._jvm.java.lang.System.gc()
    
    # 2. Альтернативный путь: вызвать GC через Runtime
    sc._jvm.java.lang.Runtime.getRuntime().gc()
    
    if verbose:
        print("✅ GC вызван")

def destroy_all_broadcasts(sc):
    """Уничтожает все зарегистрированные в трекере broadcast-блоки."""
    # Через внутренний API — работает в Spark 3.x
    try:
        jsc = sc._jsc.sc()
        # Получаем все блоки из BlockManager
        blocks = sc._jvm.org.apache.spark.storage.BlockId  # класс-маркер
        # Простой обход через Python-список (если ведётся вручную)
    except Exception:
        print("Done nothing")
    # Надёжный путь: хранить ссылки самому и вызывать .destroy()

def cleanup_python_memory():
    """Агрессивная очистка памяти в Python-процессах."""
    # 1. Принудительный сбор мусора (3 прохода)
    for _ in range(3):
        gc.collect()
    
    # 2. Очистка глобального кэша FAISS-индексов
    #    (см. get_executor_cache() в faiss.py)
    if hasattr(builtins, "_faiss_index_cache"):
        cache = builtins._faiss_index_cache
        cache._cache.clear()
        gc.collect()

def clear_all_executor_caches(
    sc,
    num_partitions: int | None = None,
) -> int:
    """Рассылает команду очистки кэша FAISS-индексов на все экзекьюторы.

    Создаёт пустой RDD с числом партиций, равным общему числу слотов
    (экзекьюторы × ядра), и запускает ``mapPartitions``, чтобы
    гарантировать выполнение хотя бы одной задачи на каждом слоте
    каждого экзекьютора.

    Args:
        sc: Активный ``SparkContext``.
        num_partitions: Число партиций для рассылки. Если ``None``,
            используется ``sc.defaultParallelism``, что обычно
            равно ``число экзекьюторов × число ядер``.

    Returns:
        Число экзекьюторных процессов, на которых была выполнена
        очистка кэша (по числу вернувшихся партиций).
    """
    def _clear_partition_cache(iterator):
        """Выполняется на каждом экзекьюторе."""
        import builtins
        import gc

        cleared = 0
        if hasattr(builtins, "_faiss_index_cache"):
            cache = builtins._faiss_index_cache
            with cache._lock:
                cache._cache.clear()
            cleared = 1
        # Явный сбор мусора для освобождения памяти,
        # занимаемой десериализованными индексами
        gc.collect()
        yield cleared

    if num_partitions is None:
        num_partitions = sc.defaultParallelism

    result = (
        sc.parallelize(range(num_partitions), num_partitions)
        .mapPartitions(_clear_partition_cache)
        .collect()
    )
    return sum(result)

def full_cleanup(session: SparkSession):
    sc = session.sparkContext
    cleanup_spark_local_dirs(sc)
    force_jvm_gc(sc)
    destroy_all_broadcasts(sc)
    cleanup_python_memory()
    clear_all_executor_caches(sc)

In [11]:
sp_s

In [19]:
full_cleanup(sp_s)

🧹 Запуск принудительного GC в JVM...
✅ GC вызван


In [14]:
dataset

,treatment,feat_num_1,feat_num_2,feat_cat,target
0,1,10.055573,0.099213,A,96.607012
1,1,9.738368,-2.477588,A,106.008497
2,0,8.56536,-5.207757,B,72.052715
3,1,11.579609,-1.766366,B,91.295394
4,1,14.548105,-3.019598,A,88.486315
...,...,...,...,...,...
9995,0,11.885614,-4.313001,C,100.428048
9996,0,7.356847,-2.061087,A,89.530951
9997,1,5.331312,-1.756478,A,99.007641
9998,0,11.684201,-3.745283,B,91.916056


In [20]:
sp_s.stop()